# Introduction
- Date: 20241218
- Last Update: 20250303
- Author: BL
- This is a script to make polished panels for Figure 3 of the spatial lung manuscript

Update Log
- Incorperated linear regression results to pseudotime
- Updated to match the newest panels

In [1]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse, stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
import warnings
from matplotlib.colors import LinearSegmentedColormap, Normalize

In [2]:
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 0

output_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/Fig2/'
sup_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/FigS2/'

os.chdir(output_loc)

# Load helpful functions and objects

In [3]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'DarkGreen',
                 'SyntheticMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'limegreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                       'ProximalIntermediate':'darkred',
                       'DistalIntermediate':'darkblue',
                       'Intermediate': 'Green',
                       'Distal':'SkyBlue'}

mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [4]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

# Load the objects

In [5]:
# Read in the full integrated adata
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

# Read in the epithelial subcluster adata
epi_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_epithelial_subcluster_pseudotime.h5ad')

# Read in the scRNA only modules identified by Monocle for the epithelial subclustered pseudotime
epi_sub_modules = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/metadata/pseudotime/epithelial_subcluster/Lung_epithelial_subcluster_scRNA_only_monocle_gene_modules_genes_above_0.1morans.txt', index_col = 0)

In [6]:
# Convert epi_sub_modules to genes
# Build dictionary for converting GeneID to Gene names
file_path_gene = '/Genomics/chanlab/pengfei/dbit/Manuscript/lungmapping/computation manual/RNAvelocity/Ensembl_geneID_conversion_v2.txt'

# Create an empty dictionary to store the data
data_dict_gene = {}

# Open the file and read its contents
with open(file_path_gene, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        key, value = line.strip().split(',')
        
        # Add key-value pairs to the dictionary
        data_dict_gene[key] = value
flipped_gene_dict = {value: key for key, value in data_dict_gene.items()}

# Build dictionary for converting GeneID to Gene names using the v3 table
file_path_gene_v3 = '/Genomics/chanlab/blaw/Spatial_Lung/metadata/Ensembl_geneID_conversion_v3.txt'

# Create an empty dictionary to store the data
data_dict_gene_v3 = {}

# Open the file and read its contents
with open(file_path_gene_v3, 'r') as file:
    for line in file:
        # Split each line into key and value using the tab separator
        value, key = line.strip().split('\t')
        
        # Add key-value pairs to the dictionary
        data_dict_gene_v3[key] = value
flipped_gene_dict_v3 = {value: key for key, value in data_dict_gene.items()}

epi_sub_modules['ensambl_id'] = epi_sub_modules['id']
for i in epi_sub_modules.index:
    if epi_sub_modules.loc[i, 'ensambl_id'] in data_dict_gene_v3.keys():
        epi_sub_modules.loc[i, 'id'] = data_dict_gene_v3[epi_sub_modules.loc[i, 'ensambl_id']]

# Figure 3b
- Spatial map of the epithelial states
- Spatial maps of Distal / Proximal marker genes: ['Sox2', 'Sox9', 'Etv5']

In [43]:
# Plot the spatial localization of just epithelial cells across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'Epithelial(Proximal)'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2b_spatial_localization_epithelial.pdf',
                    palette = clusterColors,
                    show = False)

In [11]:
# Plot the spatial localization of each gene in only epithelial cells across all replicates
genes = ['Sox2', 'Sox9', 'Etv5']

for gene in genes:        
    meta_col = gene
    palette = expr_cmap
    # Show just the Proximal cells to highligher it's expression in Proximal cells
    show_states = ['Epithelial(Distal)', 'Epithelial(Proximal)']
    size = 10
    show_size = 30
    
    temp_adata = lung_adata.copy()
    background_adata = lung_adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in temp_adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()
    
        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []
    
            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            
                    
            for state in temp_adata.obs['cellstate'].unique():
                temp_locs = temp_adata.obs[(temp_adata.obs['cellstate'].isin([state])) & (temp_adata.obs['orig.ident'] == exp)].index
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                
                if state in show_states:
                    colorbar = temp_ax.scatter(x, y, c = z, s = show_size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)
    
        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    plt.savefig(output_loc + 'Fig.3b_spatial_localization_{}_adjusted.pdf'.format(meta_col), dpi = 300)
    #plt.show()
    plt.close()

# Figure 3d
- A UMAP plot of the epithelial subclustering, colored by epithelial subcluster

In [9]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(epi_adata, color = ['subcluster_cellstate'], wspace = 0.35, palette = epiSubclusterColors, show = False)
    plt.savefig(output_loc + 'Fig.3d_epithelial_subclusters.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

# Figure 3e
- Spatial maps of the epithelial subclusters

In [44]:
# Plot the spatial localization of all cellstates across all replicates
plot_spatial_groups(adata = epi_adata,
                    meta_col = 'subcluster_cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.3e_spatial_localization_epithelial_subclusters.pdf',
                    palette = epiSubclusterColors,
                    show = False)

# Figure 3f
- Violin plots of select marker genes

In [ ]:
# Violin plots of genes in scRNA cells
genes = ['Sox2', 'Igfbp5', 'Ccnd2', 'Sox9', 'Id2', 'Sftpc', 'Icam1', 'Etv5', 'Etv4', 'Sftpb', 'Fgfbp1', 'Bmp4', 'Smoc2', 'Tbx1', 'Adamts18', 'Adamtsl1', 'Thbd']
for gene in genes:
    with rc_context({'figure.figsize': (3, 1)}):
        sc.pl.violin(epi_adata[epi_adata.obs['tech'] == 'scRNA_seq'], gene, groupby = 'subcluster_cellstate',
                     order = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal'],
                     jitter=0.4, multi_panel= True, rotation = 90, palette = epiSubclusterColors, show = False)
        plt.savefig(sup_loc + 'Fig.3f_scRNA_violin_{}.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()

# Figure 3g
- Composition plots of dbit_seq and scRNA_seq per timepoint

In [ ]:
# Plot the subcluster composition per timepoint in scRNA cells
time_composition = pd.DataFrame(0, columns = epi_subcluster_cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(epi_adata.obs[(epi_adata.obs['timepoint'] == i) & (epi_adata.obs['subcluster_cellstate'] == j) & (epi_adata.obs['tech'] == 'scRNA_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = epiSubclusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint scRNA')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.3g_epithelial_subcluster_composition_per_timepoint_scRNA.pdf', dpi = 300)
#plt.show()
plt.close()

In [ ]:
# Plot the subcluster composition per timepoint in DBiT cells
time_composition = pd.DataFrame(0, columns = epi_subcluster_cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(epi_adata.obs[(epi_adata.obs['timepoint'] == i) & (epi_adata.obs['subcluster_cellstate'] == j) & (epi_adata.obs['tech'] == 'DBiT_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = epiSubclusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint DBiT')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.3g_epithelial_subcluster_composition_per_timepoint_DBiT.pdf', dpi = 300)
#plt.show()
plt.close()

# Figure 3h
- DE results of DI marker genes

In [45]:
# Plot a volcano plot of Distal vs DI
upName = 'DistalIntermediate'
downName = 'not_{}'.format(upName)
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/marker_genes/{}_vs_rest.csv'.format(upName), index_col = 0)

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Sox2', 'Sox9', 'Icam1', 'Fgfbp1', 'Etv5', 'Id2', 'Bmp4', 'Smoc2', 'Adamtsl1', 'Foxp2']

fig, ax = plt.subplots(figsize = (5, 5))
#plt.style.use('default')

# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['logfoldchanges'].values
y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')

# highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
x_up = []
y_up = []
x_down = []
y_down = []
for count, val in enumerate(x):
    if x[count] > 1 and y[count] > 1.3:
        x_up.append(x[count])
        y_up.append(y[count])
plt.scatter(x = x_up, y = y_up, s = 10, color = epiSubclusterColors[upName])

for gene in annotate_genes:
    x = temp_df.loc[gene, 'logfoldchanges']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    plt.text(x, y + 0.5, gene) 

ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlim(-2, 2)
plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('{} vs {} - DE'.format(upName, downName))
plt.savefig(output_loc + 'Fig.3h_{}_volcano_plot.pdf'.format(upName), dpi = 300)
plt.show()
plt.close()

# Figure 3i
- A heatmap of marker genes across subclusters

In [ ]:
# Collect the top genes from each DE comparison
top_genes = []

for comparison in ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']:
    temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/marker_genes/{}_vs_rest.csv'.format(comparison), index_col = 0)
    
    for gene in temp_df[(temp_df['pvals'] < 0.05) & (temp_df['manualLogFC'] > 0.75)][0:5].index:
        if gene not in top_genes:
            top_genes.append(gene)

In [ ]:
# create a dataframe of the zscore of the average expression per subcluster
expr_df = pd.DataFrame(columns = epi_adata.var.index, index = epi_subcluster_cellstates)

for i in expr_df.index:
    expr_df.loc[i] = epi_adata[(epi_adata.obs['subcluster_cellstate'] == i)].X.mean(axis = 0)
    
# calculate the zscores of genes
z_expr_df = expr_df.copy()

# remove genes that have a std of 0 (this will break the z score)
bad_genes = []
for gene in z_expr_df.columns:
    if z_expr_df[gene].std() == 0:
        bad_genes.append(gene)  
z_expr_df.drop(columns = bad_genes, inplace = True)

mean_vals = z_expr_df.mean(axis = 0)
std_vals = z_expr_df.std(axis = 0)
z_expr_df = (z_expr_df - mean_vals) / (std_vals)
z_expr_df.fillna(0, inplace = True)

# Plot the zscore dataframe using the fol/lowing genes:
temp = z_expr_df.astype('float').copy()
# Manually reorder genes to show Distal to Proximal change
genes = ['Bex4', 'Dlk1', 'Clu', 'Sox9', 'Etv5', 'Smoc2', 'Ckb', 'Cldn10', 'Tgfb3', 'Tbx1', 'Ccnd2', 'Igfbp5', 'Anxa2', 'Sox2', 'Adamtsl1']

drop_col = []
for gene in temp.columns:
    if gene not in genes:
        drop_col.append(gene)
temp.drop(columns = drop_col, inplace = True)
colormap = cm.RdYlBu_r

g = sns.clustermap(temp[genes].T, figsize = (3, 6), col_cluster = False, row_cluster = False, cmap=colormap, vmin = -2, vmax=2)
plt.subplots_adjust(bottom = 0.3, right = 0.7)
plt.savefig(output_loc + 'Fig.3i_subcluster_DE_heatmap.pdf', dpi = 300)
#plt.show()
plt.close()

# Figure 3j and 3k
- Spatial maps of plots of several key genes from the DE and module analysis: ['Smoc2', 'Adamtsl1']

In [8]:
# Plot the spatial localization of each gene across all replicates
genes = ['Smoc2', 'Adamtsl1']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = output_loc,
                        name = 'Fig.2l_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

In [9]:
new_label = []

for i in lung_adata.obs.index:
    if i in epi_adata.obs.index:
        new_label.append(epi_adata.obs.loc[i, 'subcluster_cellstate'])
    else:
        new_label.append('-')

lung_adata.obs['subcluster_cellstate'] = new_label

In [14]:
# Plot spatial maps of the expression of these genes making epithelial pixals bigger
meta_col = 'Smoc2'
palette = expr_cmap
# Show just the proximal intermediate and distal intermediate states to highlight smoc2
show_states = ['ProximalIntermediate', 'DistalIntermediate']
size = 10
show_size = 30

temp_adata = lung_adata.copy()
background_adata = lung_adata.copy()

# Add the gene expr values to the obs table if plotting a gene
if meta_col in temp_adata.var.index:
    temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()

timepoints = ['E11.5', 'E12.5', 'E13.5']
fig, axs = plt.subplots(3, 5, figsize = (35, 21))
row_count = 0
for time in timepoints:
    datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

    col_count = 0
    for exp in datasets:
        temp_ax = axs[row_count, col_count]
        col_count += 1
        
        # Plot full structure in gray from the background adata
        all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
        x = []
        y = []

        for i in all_cell_locs:
            y.append(background_adata.obs.loc[i, 'y_loc'])
            x.append(background_adata.obs.loc[i, 'x_loc'])
        # Use silver with a default alpha to match the lower bound of the expression plots
        temp_ax.scatter(x, y, color = 'silver', s = size)
    
        # plot the quantitative variable
        
                
        for state in temp_adata.obs['subcluster_cellstate'].unique():
            temp_locs = temp_adata.obs[(temp_adata.obs['subcluster_cellstate'].isin([state])) & (temp_adata.obs['orig.ident'] == exp)].index
            x = [0]
            y = [0]
            z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
            for i in temp_locs:
                z.append(temp_adata.obs.loc[i, meta_col])
                y.append(temp_adata.obs.loc[i, 'y_loc'])
                x.append(temp_adata.obs.loc[i, 'x_loc'])
            
            if state in show_states:
                colorbar = temp_ax.scatter(x, y, c = z, s = show_size, cmap = palette)
            else:
                colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
        temp_ax.set_title('{}_{}'.format(exp, meta_col))
        temp_ax.set_xlabel('X Pos')
        temp_ax.set_ylabel('Y Pos')
        temp_ax.set_xlim(0, 52)
        temp_ax.set_ylim(0, 52)

    row_count += 1
if type(temp_adata.obs[meta_col][0]) != str:
    fig.colorbar(colorbar, ax = axs[2, 4])
plt.savefig(output_loc + 'Fig.3k_spatial_localization_{}_adjusted.pdf'.format(meta_col), dpi = 300)
#plt.show()
plt.close()

In [10]:
# Plot spatial maps of the expression of these genes making epithelial pixals bigger
meta_col = 'Adamtsl1'
palette = expr_cmap
# Show just the Proximal cells to highligher it's expression in Proximal cells
show_states = ['Proximal']
size = 10
show_size = 30

temp_adata = lung_adata.copy()
background_adata = lung_adata.copy()

# Add the gene expr values to the obs table if plotting a gene
if meta_col in temp_adata.var.index:
    temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()

timepoints = ['E11.5', 'E12.5', 'E13.5']
fig, axs = plt.subplots(3, 5, figsize = (35, 21))
row_count = 0
for time in timepoints:
    datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

    col_count = 0
    for exp in datasets:
        temp_ax = axs[row_count, col_count]
        col_count += 1
        
        # Plot full structure in gray from the background adata
        all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
        x = []
        y = []

        for i in all_cell_locs:
            y.append(background_adata.obs.loc[i, 'y_loc'])
            x.append(background_adata.obs.loc[i, 'x_loc'])
        # Use silver with a default alpha to match the lower bound of the expression plots
        temp_ax.scatter(x, y, color = 'silver', s = size)
    
        # plot the quantitative variable
        
                
        for state in temp_adata.obs['subcluster_cellstate'].unique():
            temp_locs = temp_adata.obs[(temp_adata.obs['subcluster_cellstate'].isin([state])) & (temp_adata.obs['orig.ident'] == exp)].index
            x = [0]
            y = [0]
            z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
            for i in temp_locs:
                z.append(temp_adata.obs.loc[i, meta_col])
                y.append(temp_adata.obs.loc[i, 'y_loc'])
                x.append(temp_adata.obs.loc[i, 'x_loc'])
            
            if state in show_states:
                colorbar = temp_ax.scatter(x, y, c = z, s = show_size, cmap = palette)
            else:
                colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
        temp_ax.set_title('{}_{}'.format(exp, meta_col))
        temp_ax.set_xlabel('X Pos')
        temp_ax.set_ylabel('Y Pos')
        temp_ax.set_xlim(0, 52)
        temp_ax.set_ylim(0, 52)

    row_count += 1
if type(temp_adata.obs[meta_col][0]) != str:
    fig.colorbar(colorbar, ax = axs[2, 4])
plt.savefig(output_loc + 'Fig.3j_spatial_localization_{}_adjusted.pdf'.format(meta_col), dpi = 300)
#plt.show()
plt.close()

# Extended Data
- Spatial maps of Proximal and Distal Epithelial cells
- Spatial maps of Epithelial markers, highlighting epithelial cells: ['Sox9', 'Id2', 'Etv5']
- Spatial maps of Epithelial subclusters
- Histograms of Epithelial subcluster populations over time
- Composition of Epithelial cells per population of Sox9 / Sox2 positive cells
- Composition of Epithelial subclusters in scRNA replicates
- Composition of Epithelial subclusters in DBiT replicates
- Volcano plots for DE comparisons (PI vs Proximal, DI vs Distal)
- Violin plots of subcluster marker genes: ['Sox2', 'Igfbp5', 'Ccnd2', 'Sox9', 'Id2', 'Sftpc', 'Icam1', 'Etv5', 'Sftpb', 'Fgfbp1', 'Bmp4', 'Smoc2', 'Tbx1', 'Adamts18', 'Adamtsl1', 'Thbd']
- Spatial plots of subcluster markers using the same size marker: ['Bmp4', 'Icam1', 'Adamts18', 'Sftpb', 'Thbd']

In [8]:
os.chdir(sup_loc)

In [46]:
# Plot spatial maps of epithelial subclusters
plot_spatial_groups(adata = epi_adata,
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S3_spatial_localization_epithelial.pdf',
                    palette = clusterColors,
                    show = False)

In [21]:
# Plot the spatial localization of each gene across all replicates
genes = ['Sox9', 'Id2', 'Etv5', 'Bmp4', 'Icam1', 'Adamts18', 'Sftpb', 'Thbd']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = sup_loc,
                        name = 'Fig.S3_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

In [10]:
# Plot spatial maps of the expression of these genes making epithelial pixals bigger
genes = ['Sox2', 'Sox9', 'Id2', 'Etv5', 'Bmp4', 'Icam1', 'Adamts18', 'Sftpb', 'Thbd']

for gene in genes:        
    meta_col = gene
    palette = expr_cmap
    # Show just the Proximal cells to highligher it's expression in Proximal cells
    show_states = ['Epithelial(Distal)', 'Epithelial(Proximal)']
    size = 10
    show_size = 30
    
    temp_adata = lung_adata.copy()
    background_adata = lung_adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in temp_adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()
    
        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []
    
            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            
                    
            for state in temp_adata.obs['cellstate'].unique():
                temp_locs = temp_adata.obs[(temp_adata.obs['cellstate'].isin([state])) & (temp_adata.obs['orig.ident'] == exp)].index
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                
                if state in show_states:
                    colorbar = temp_ax.scatter(x, y, c = z, s = show_size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)
    
        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    plt.savefig(sup_loc + 'Fig.S3_spatial_localization_{}_adjusted.pdf'.format(meta_col), dpi = 300)
    #plt.show()
    plt.close()

In [47]:
# Plot spatial maps of epithelial subclusters
plot_spatial_groups(adata = epi_adata,
                    meta_col = 'subcluster_cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S3_spatial_localization_epithelial_subcluster.pdf',
                    palette = epiSubclusterColors,
                    show = False)

In [23]:
hist_maxs = {'Distal': 200,
             'DistalIntermediate': 200,
             'ProximalIntermediate': 150,
             'Proximal': 150}

# For each subcluster celltype, plot a histogram of each timepoint to show it's progression along the pseudotime
for state in epi_subcluster_cellstates:
    fig, axs = plt.subplots(3, 1, figsize = (6, 9))
    
    labels = []
    for i, timepoint in enumerate(timepoints):
        temp_ax = axs[i]
        temp_values = epi_adata.obs[(epi_adata.obs['timepoint'] == timepoint) & (epi_adata.obs['subcluster_cellstate'] == state)]['epithelial_pseudotime']
        if len(temp_values) > 1:
            sns.histplot(temp_values, ax = temp_ax, bins = [i for i in range(0, 26)], color = epiSubclusterColors[state])
        temp_ax.set_title('{}_{}'.format(state, timepoint))
        temp_ax.set_xlim(0, 26)
        temp_ax.set_ylim(0, hist_maxs[state])
    
    plt.tight_layout()
    plt.savefig(sup_loc + 'Fig.S3_{}_epithelial_pseudotime_hist.pdf'.format(state), dpi = 300)
    #plt.show()
    plt.close()

In [13]:
# Plot a volcano plot of Distal vs DI
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/gene_lists/epithelial_subclusters_scRNA_only_Distal_DistalIntermediate_wilcox_DE.csv', index_col = 0)
upName = 'Distal'
downName = 'DistalIntermediate'

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Bex4', 'Dlk1', 'Smoc2', 'Etv5', 'Sftpc', 'Aldh1a7', 'Adamtsl1', 'Sox2', 'Tbx1', 'Sox9', 'Igfbp5', 'Id2', 'Bmp4', 'Adamts18', 'Etv4']

fig, ax = plt.subplots(figsize = (5, 5))
#plt.style.use('default')

# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')

# highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
x_up = []
y_up = []
x_down = []
y_down = []
for count, val in enumerate(x):
    if x[count] > 1 and y[count] > 1.3:
        x_up.append(x[count])
        y_up.append(y[count])
    if x[count] < -1 and y[count] > 1.3:
        x_down.append(x[count])
        y_down.append(y[count])
plt.scatter(x = x_up, y = y_up, s = 10, color = epiSubclusterColors[upName])
plt.scatter(x = x_down, y = y_down, s = 10, color = epiSubclusterColors[downName])

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    plt.text(x, y + 0.5, gene) 

ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('{} vs {} - Differential Expression'.format(upName, downName))
plt.savefig(sup_loc + 'Fig.S3_{}_vs_{}_volcano_plot.pdf'.format(upName, downName), dpi = 300)
#plt.show()
plt.close()

In [24]:
# Plot a volcano plot of DI vs PI
temp_df = pd.read_csv('/Genomics/chanlab/blaw/Spatial_Lung/sandbox/scRNA/DE/epithelial_subclusters/gene_lists/epithelial_subclusters_scRNA_only_ProximalIntermediate_DistalIntermediate_wilcox_DE.csv', index_col = 0)
upName = 'ProximalIntermediate'
downName = 'DistalIntermediate'

# add pseudolowest pvals for any that are too low for the computer
temp_df['pvals_adj_2'] = temp_df['pvals_adj']
for gene in temp_df[temp_df['pvals_adj'] == 0].index:
    temp_df.loc[gene, 'pvals_adj_2'] = temp_df[temp_df['pvals_adj'] > 0]['pvals_adj'].min()

annotate_genes = ['Smoc2', 'Etv5', 'Sftpc', 'Aldh1a7', 'Adamtsl1', 'Sox2', 'Sparc', 'Thbs1', 'Sox9', 'Igfbp5', 'Fstl1', 'Cdh2', 'Tbx1', 'Foxp2', 'Adamts18', 'Lin7a']

fig, ax = plt.subplots(figsize = (5, 5))
#plt.style.use('default')

# Plot the volcano plot of the differential expression using manual LogFC
x = temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['manual_LogFC'].values
y = -np.log10(temp_df[(temp_df['{}_avg_expr'.format(upName)] != 0) & (temp_df['{}_avg_expr'.format(downName)] != 0)]['pvals_adj_2']).values
plt.scatter(x = x, y = y, alpha = 0.3, s = 5, color = 'lightgray')

# highlight genes that are higher than 1 logfoldchange and higher than 1.3 -log10 adjustedpval
x_up = []
y_up = []
x_down = []
y_down = []
for count, val in enumerate(x):
    if x[count] > 1 and y[count] > 1.3:
        x_up.append(x[count])
        y_up.append(y[count])
    if x[count] < -1 and y[count] > 1.3:
        x_down.append(x[count])
        y_down.append(y[count])
plt.scatter(x = x_up, y = y_up, s = 10, color = epiSubclusterColors[upName])
plt.scatter(x = x_down, y = y_down, s = 10, color = epiSubclusterColors[downName])

for gene in annotate_genes:
    x = temp_df.loc[gene, 'manual_LogFC']
    y = -np.log10(temp_df.loc[gene, 'pvals_adj_2'])
    plt.text(x, y + 0.5, gene) 

ax.axhline(1.3, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.axvline(-1, zorder = 0, c = 'gray', lw = 1, ls = '--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlabel('LogFC')
plt.ylabel('-log10(adj_pval)')
plt.title('{} vs {} - Differential Expression'.format(upName, downName))
plt.savefig(sup_loc + 'Fig.S3_{}_vs_{}_volcano_plot.pdf'.format(upName, downName), dpi = 300)
#plt.show()
plt.close()